# Debug the intervention feature

In [ ]:
!git clone https://github.com/mGarbowski/zzsn-projekt.git
!cd zzsn-projekt && git checkout intervention

In [ ]:
import os
os.chdir('zzsn-projekt')
os.getcwd()

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from diffusers import StableDiffusionPipeline
model_id = "CompVis/stable-diffusion-v1-4"
dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype)
pipe = pipe.to(device)
pipe.set_progress_bar_config(disable=True)

In [ ]:
from models.linear import SchmidhuberLinear, SchmidhuberLinearConfig
loaded_state = torch.load("../example_weights.pt", map_location="cpu")
loaded_state.keys()
cfg = SchmidhuberLinearConfig(
    expansion_factor=2,
    input_dim=1280,
    predictor_dropout=0.3,
    predictor_embedding_dim=128,
    predictor_hidden_dims=[256]
)
schmidhuber = SchmidhuberLinear(cfg)
schmidhuber.load_state_dict(loaded_state)
schmidhuber = schmidhuber.to(device).to(dtype)

In [ ]:
from models.diffusion import WrappedDiffusion, GenerationParams
wrapped = WrappedDiffusion(pipe, schmidhuber)
params =GenerationParams(
    prompt="A picture of grey british shorthair cat",
    num_inference_steps=50,
    guidance_scale=7.5
)

## Image without intervention

In [ ]:
wrapped.diffusion(params.prompt, num_inference_steps=params.num_inference_steps,guidance_scale=params.guidance_scale)

## Image with intervention

In [ ]:
intervention_multipliers = {
    key: 0 for key in range(10, 50)
}
res = wrapped.generate_with_intervention(params, intervention_multipliers)

In [ ]:
res.images[0]